In [179]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# from sklearn.model_selection import GridSearchCV

from scipy.stats import norm
from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix, roc_curve, auc, recall_score, precision_score, f1_score


In [180]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# from sklearn.model_selection import GridSearchCV

from scipy.stats import norm
from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix, roc_curve, auc


# 1. Load dataset

In [181]:

data_path = Path(r"..\Healthcare\dataset\heart.csv")
if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found at {data_path}")


df = pd.read_csv(data_path)
print("Dataset loaded from:", data_path)
print("\nShape:", df.shape)

Dataset loaded from: ..\Healthcare\dataset\heart.csv

Shape: (1025, 14)


# 2. Quick look and check for Missing values and basic stats

In [182]:
df.head ()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0


In [183]:
print("\nMissing values per column:")
print(df.isna().sum())


Missing values per column:
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64


In [184]:
dupvalues = df.duplicated().any()
if dupvalues:
    no_of_dup = df.duplicated().sum()
    print(f"\nThere are {no_of_dup} duplicate rows in the dataset.")
else:
    print("\nNo duplicate rows found in the dataset.")


There are 723 duplicate rows in the dataset.


Dropping Duplicate Rows and assigining to variable name nd 

In [185]:
nd_df = df.drop_duplicates()
nd_df.shape

(302, 14)

In [186]:

nd_df.info()
nd_df.describe()

<class 'pandas.core.frame.DataFrame'>
Index: 302 entries, 0 to 878
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       302 non-null    int64  
 1   sex       302 non-null    int64  
 2   cp        302 non-null    int64  
 3   trestbps  302 non-null    int64  
 4   chol      302 non-null    int64  
 5   fbs       302 non-null    int64  
 6   restecg   302 non-null    int64  
 7   thalach   302 non-null    int64  
 8   exang     302 non-null    int64  
 9   oldpeak   302 non-null    float64
 10  slope     302 non-null    int64  
 11  ca        302 non-null    int64  
 12  thal      302 non-null    int64  
 13  target    302 non-null    int64  
dtypes: float64(1), int64(13)
memory usage: 35.4 KB


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
count,302.00000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000
mean,54.42053,0.682119,0.963576,131.602649,246.500000,0.149007,0.526490,149.569536,0.327815,1.043046,1.397351,0.718543,2.314570,0.543046
std,9.04797,0.466426,1.032044,17.563394,51.753489,0.356686,0.526027,22.903527,0.470196,1.161452,0.616274,1.006748,0.613026,0.498970
min,29.00000,0.000000,0.000000,94.000000,126.000000,0.000000,0.000000,71.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,48.00000,0.000000,0.000000,120.000000,211.000000,0.000000,0.000000,133.250000,0.000000,0.000000,1.000000,0.000000,2.000000,0.000000
50%,55.50000,1.000000,1.000000,130.000000,240.500000,0.000000,1.000000,152.500000,0.000000,0.800000,1.000000,0.000000,2.000000,1.000000
75%,61.00000,1.000000,2.000000,140.000000,274.750000,0.000000,1.000000,166.000000,1.000000,1.600000,2.000000,1.000000,3.000000,1.000000
max,77.00000,1.000000,3.000000,200.000000,564.000000,1.000000,2.000000,202.000000,1.000000,6.200000,2.000000,4.000000,3.000000,1.000000


# 4. Target distribution

In [187]:
# Identified our target column as "target"
print("\nTarget value counts:")
print(df["target"].value_counts(dropna=False))


Target value counts:
target
1    526
0    499
Name: count, dtype: int64


In [188]:
out_dir = Path(r"..\Healthcare\analysis_output")
out_dir.mkdir(exist_ok=True)

# 5.  Correlation matrix Heat map

In [189]:
numeric_df = nd_df.select_dtypes(include=[np.number])
corr = numeric_df.corr()

# plt.figure(figsize=(8,6))
# plt.imshow(corr, aspect='auto')
# plt.colorbar()
# plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
# plt.yticks(range(len(corr.columns)), corr.columns)
# plt.tight_layout()
# plt.savefig(out_dir / "correlaation_matrix.png", bbox_inches='tight')
# plt.close()

plt.figure(figsize=(12,6))
plt.title("Correlation matrix Heat map")
sns.heatmap(df.corr(),annot=True)
plt.savefig(out_dir / "heatmap.png", bbox_inches='tight')
# plt.show()
plt.close()

In [190]:
sns.countplot(x ='target',data=nd_df)
plt.xticks([0, 1], ['Without Heart Diseases', 'With Heart Diseases'])
plt.title('Distribution of Heart Disease')
plt.xlabel('Heart Disease (0 = No, 1 = Yes)')
plt.ylabel('Count')
plt.savefig(out_dir / "Heart Disesase dist.png", bbox_inches='tight')
# plt.show()
plt.close()

gender distribution

In [191]:
sns.countplot(x='sex', data=nd_df)    
plt.title('Distribution of Gender ')
plt.xticks([0, 1], ['Female', 'Male'])
plt.xlabel('Gender (0 = Female, 1 = Male)')
plt.ylabel('Count')
plt.savefig(out_dir / "gender dist.png", bbox_inches='tight')
# plt.show()
plt.close()

Distribution of heart diseases among males and females 

In [192]:
sns.countplot(x='sex', hue='target', data=nd_df)
plt.xticks([0, 1], ['Female', 'Male'])
plt.title('Distribution of Heart Disease by Gender')
plt.legend(title='Heart Disease', labels=['No Heart diseases', 'with Heart diseases'])
plt.xlabel('Gender (0 = Female, 1 = Male)')
plt.ylabel('Count') 
plt.savefig(out_dir / "dist among gender.png", bbox_inches='tight')
# plt.show()
plt.close()

Distribution of heart diseases by age 

In [193]:
from matplotlib.pyplot import hist


sns.displot( nd_df['age'],bins = 20, kde =True)
# plt.xticks([0, 1], ['Female', 'Male'])
# plt.title('Distribution of Heart Disease by Gender')
# plt.legend(title='Heart Disease', labels=['No Heart diseases', 'with Heart diseases'])
# plt.xlabel('Gender (0 = Female, 1 = Male)')
# plt.ylabel('Count') 
plt.savefig(out_dir / "dist by age.png", bbox_inches='tight')
# plt.show()
plt.close()

Chest Pain Type

In [194]:
sns.countplot(x='cp', data=nd_df)    
plt.title('Distribution chest pain tyoe  ')
plt.xticks([0, 1, 2, 3], ['typical angina', 'atypical angina', 'non-anginal pain', 'asymptomatic'])
plt.xlabel('Chest Pain Type')
plt.ylabel('Count')
plt.savefig(out_dir / "chest pain dist.png", bbox_inches='tight')
# plt.show()
plt.close()

chest pain by heart diseases

In [195]:
sns.countplot(data = nd_df, x='cp', hue='target')
plt.title('Chest Pain Type by Heart Disease')
plt.legend(title='Heart Disease', labels=['No Heart diseases', 'with Heart diseases'])
plt.xticks([0, 1, 2, 3], ['typical angina', 'atypical angina', 'non-anginal pain', 'asymptomatic'])
plt.xlabel('Chest Pain Type')
plt.ylabel('Count')
plt.savefig(out_dir / "dist by chest pain.png", bbox_inches='tight')
# plt.show()
plt.close()

Distribution by blood presure levels

In [196]:
nd_df['trestbps'].hist(edgecolor='black')
plt.title('Distribution of Resting Blood Pressure')
plt.savefig(out_dir / "bist by blood pressure.png", bbox_inches='tight')
# plt.show()
plt.close()

Distribution by serum cholesterol

In [197]:
nd_df['chol'].hist(edgecolor='black')
plt.title('Distribution of Cholesterol Levels')
plt.savefig(out_dir / "dist by cholesterol.png", bbox_inches='tight')
# plt.show()
plt.close()

# Training the machine learning algo

Train test split 

In [198]:
x, y = nd_df.drop('target', axis=1), nd_df['target']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=9)
print("\nTrain shape:", x_train.shape, "Test shape:", x_test.shape)


Train shape: (241, 13) Test shape: (61, 13)


 11. Train Random Forest

Scale Insensitive Models 

In [200]:
rf = RandomForestClassifier()
rf.fit(x_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [201]:
nb_clf = GaussianNB()
nb_clf.fit(x_train, y_train)

,priors,None
,var_smoothing,1e-09


In [202]:
gb_clf = GradientBoostingClassifier()
gb_clf.fit(x_train, y_train)

,loss,'log_loss'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


Scale Sensitive Models 

In [203]:
scaler = StandardScaler()
x_train_scaled= scaler.fit_transform(x_train)
x_test_scaled= scaler.transform(x_test)

In [204]:
knn = KNeighborsClassifier()
knn.fit(x_train_scaled, y_train)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [205]:
log = LogisticRegression()
log.fit(x_train_scaled, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [206]:
svc = SVC()
svc.fit(x_train_scaled, y_train)

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


Evaluate Models 

In [207]:
rf.score(x_test, y_test)

0.8032786885245902

In [208]:
nb_clf.score(x_test, y_test)

0.8032786885245902

In [209]:
gb_clf.score(x_test, y_test)

0.7213114754098361

In [210]:
knn.score(x_test_scaled, y_test)

0.819672131147541

In [211]:
log.score(x_test_scaled, y_test)

0.8032786885245902

In [212]:
svc.score(x_test_scaled, y_test)

0.7868852459016393

evaluating the recall score for the models 

In [213]:
y_pred_rf = rf.predict(x_test)
y_pred_nb = nb_clf.predict(x_test)
y_pred_gb = gb_clf.predict(x_test)
y_pred_knn = knn.predict(x_test_scaled)
y_pred_log = log.predict(x_test_scaled)
y_pred_svc = svc.predict(x_test_scaled)


In [214]:
print('Random Forest',recall_score(y_test, y_pred_rf))
print('Naive Bayes',recall_score(y_test, y_pred_nb))
print('Gradient Boosting',recall_score(y_test, y_pred_gb))
print('K-Nearest Neighbors',recall_score(y_test, y_pred_knn))
print('Logistic Regression',recall_score(y_test, y_pred_log))
print('Support Vector Classifier',recall_score(y_test, y_pred_svc))


Random Forest 0.8157894736842105
Naive Bayes 0.8157894736842105
Gradient Boosting 0.6842105263157895
K-Nearest Neighbors 0.8157894736842105
Logistic Regression 0.7894736842105263
Support Vector Classifier 0.7894736842105263


In [215]:
# Random Forest metrics
rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf)
rf_recall = recall_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)

# Naive Bayes metrics
nb_clf_accuracy = accuracy_score(y_test, y_pred_nb)
nb_clf_precision = precision_score(y_test, y_pred_nb)
nb_clf_recall = recall_score(y_test, y_pred_nb)
nb_clf_f1 = f1_score(y_test, y_pred_nb)

# Gradient Boosting metrics
gb_clf_accuracy = accuracy_score(y_test, y_pred_gb)
gb_clf_precision = precision_score(y_test, y_pred_gb)
gb_clf_recall = recall_score(y_test, y_pred_gb)
gb_clf_f1 = f1_score(y_test, y_pred_gb)

# K-Nearest Neighbors metrics
knn_accuracy = accuracy_score(y_test, y_pred_knn)
knn_precision = precision_score(y_test, y_pred_knn)
knn_recall = recall_score(y_test, y_pred_knn)
knn_f1 = f1_score(y_test, y_pred_knn)

# Logistic Regression metrics
log_accuracy = accuracy_score(y_test, y_pred_log)
log_precision = precision_score(y_test, y_pred_log)
log_recall = recall_score(y_test, y_pred_log)
log_f1 = f1_score(y_test, y_pred_log)

# Support Vector Classifier metrics
svc_accuracy = accuracy_score(y_test, y_pred_svc)
svc_precision = precision_score(y_test, y_pred_svc)
svc_recall = recall_score(y_test, y_pred_svc)
svc_f1 = f1_score(y_test, y_pred_svc)


In [216]:
# Compile results into a DataFrame
from tkinter import N


results = pd.DataFrame({
    "Model": ["Random Forest","Naive Bayes","Gradient Boosting","K-Nearest Neighbors","Logistic Regression","Support Vector Classifier"],
    "Accuracy": [rf_accuracy, nb_clf_accuracy, gb_clf_accuracy, knn_accuracy, log_accuracy, svc_accuracy],
    "Precision": [rf_precision, nb_clf_precision, gb_clf_precision, knn_precision, log_precision, svc_precision],
    "Recall (Sensitivity)": [rf_recall, nb_clf_recall, gb_clf_recall, knn_recall, log_recall, svc_recall],
    "F1-score": [rf_f1,nb_clf_recall,gb_clf_f1,knn_f1,log_f1,svc_f1]
})

print("\nModel Performance Comparison:")
print(results.round(3))



Model Performance Comparison:
                       Model  Accuracy  Precision  Recall (Sensitivity)  \
0              Random Forest     0.803      0.861                 0.816   
1                Naive Bayes     0.803      0.861                 0.816   
2          Gradient Boosting     0.721      0.839                 0.684   
3        K-Nearest Neighbors     0.820      0.886                 0.816   
4        Logistic Regression     0.803      0.882                 0.789   
5  Support Vector Classifier     0.787      0.857                 0.789   

   F1-score  
0     0.838  
1     0.816  
2     0.754  
3     0.849  
4     0.833  
5     0.822  
